### **Apply High Density Montage**

In [ ]:
# High Density Montage

import sys
import mne
import numpy as np
from pathlib import Path



CHANNEL_RENAME_MAP = {**{str(i): f'E{i}' for i in range(1, 281)}, 'REF CZ': 'Cz'}

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def log(msg: str):
    print(msg)

def parse_gpsc(filepath: Path):
    channels = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                try:
                    name = parts[0]
                    x, y, z = map(float, parts[1:4])
                    channels.append((name, x, y, z))
                except ValueError:
                    continue
    return channels

def apply_channel_renaming(raw: mne.io.Raw) -> mne.io.Raw:
    existing_map = {old: new for old, new in CHANNEL_RENAME_MAP.items() if old in raw.ch_names}
    if existing_map:
        raw.rename_channels(existing_map)
        log(f"Renamed {len(existing_map)} channels.")
    return raw

def apply_montage(raw: mne.io.Raw, gpsc_file: Path) -> mne.io.Raw:
    channels = parse_gpsc(gpsc_file)
    if not channels:
        raise ValueError("No valid channels in .gpsc file")
    gpsc_array = np.array([ch[1:4] for ch in channels])
    mean_pos = gpsc_array.mean(axis=0)
    ch_pos = {
        ch[0]: np.array([ch[1] - mean_pos[0], ch[2] - mean_pos[1], ch[3] - mean_pos[2]]) / 1000.0
        for ch in channels
    }
    montage = mne.channels.make_dig_montage(
        ch_pos=ch_pos,
        nasion=ch_pos.get('FidNz'),
        lpa=ch_pos.get('FidT9'),
        rpa=ch_pos.get('FidT10'),
        coord_frame='head'
    )
    raw.set_montage(montage, on_missing='warn')
    log("Montage applied.")
    return raw
